# Silver Layer: Data Quality & Business Logic Transformation

In [0]:
import logging
# Set up pipeline logging so failures are trackable 
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('SilverLayer')
logger.info('Initializing Silver Layer Transformation.')
# to work in particular catalog and schema
spark.sql('USE CATALOG ecom_catalog')
spark.sql('USE silver')


INFO:SilverLayer:Initializing Silver Layer Transformation.
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p0


DataFrame[]

#  Extract from Bronze

In [0]:
# Read the incremental Bronze tables
df_cart = spark.table('ecom_catalog.bronze.cart_bronze')
df_products = spark.table('ecom_catalog.bronze.products_bronze')




# ### 2. Cart Data Quality Enforcement
We apply defensive programming here. Even though the Bronze load *should* be clean, we force a deduplication step to protect our downstream `MERGE` from failing if the upstream source system accidentally sends identical rows.

In [0]:
from pyspark.sql.functions import col, current_timestamp
# to identify any duplicate are there in cart table
df_duplicates = df_cart.groupBy("cart_id", "product_id").count().filter("count > 1")
display(df_duplicates)




INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Python Server ready to receive messages
INFO:py4j.clientserver:Received command c on object id p0


cart_id,product_id,count


In [0]:
#Deduplication (Compound Key)
df_cart = df_cart.dropDuplicates(["cart_id", "product_id"])
# Handle missing financial metrics
df_cart = df_cart.fillna({
"quantity": 0,
"price": 0.0,
"total": 0.0
})
#  Enforce strict schema types
df_cart = df_cart.withColumn("cart_id", col("cart_id").cast("int")).withColumn("user_id", col("user_id").cast("int")).withColumn("product_id", col("product_id").cast("int")).withColumn("price", col("price").cast("double")).withColumn("discount_percentage", col("discount_percentage").cast("double")).withColumn("total", col("total").cast("double")).withColumn("discounted_total", col("discounted_total").cast("double"))\
.withColumn("total", col("total").cast("double")).withColumn("discounted_total", col("discounted_total").cast("double"))

# to add ingest time for tacking the time 
df_cart = df_cart.withColumn("ingest_time", current_timestamp())
logger.info("Cart data quality checks passed.")

INFO:SilverLayer:Cart data quality checks passed.


# Product Dimension Cleaning

In [0]:
from pyspark.sql.functions import col, trim, current_timestamp, lit
# smarter deduplication (Allows price changes to pass through)
df_products = df_products.dropDuplicates(["product_id", "price"]) # drop only when priceand id matches beacuse we have to save older prizes also and newer price also

# handle missing dimension attributes
df_products = df_products.fillna({
"brand": "Unknown",
"category": "Uncategorized"
})
# standardization of product title and category
df_products = df_products.withColumn("title", trim(col("title"))).withColumn("category", trim(col("category")))

#enforce strict schema types AND add SCD Type 2 tracking columns
df_products = df_products.withColumn("product_id", col("product_id").cast("int")).withColumn("price", col("price").cast("double")).withColumn("valid_from", current_timestamp()).withColumn("valid_to", lit(None).cast("timestamp")).withColumn("is_current", lit(True))

logger.info("Product dimensions cleaned, standardized, and prepped for SCD2.")

INFO:SilverLayer:Product dimensions cleaned, standardized, and prepped for SCD2.


# STATISTICS USED  Advanced Business Logic: "HIGH_Value and Normal_value order detection|'  (IQR)

In [0]:
from pyspark.sql.functions import mean, stddev, trim

In [0]:
from pyspark.sql.functions import col, when

# calculate the 25th (Q1) and 75th (Q3) percentiles
quantiles = df_cart.approxQuantile("total", [0.25, 0.75], 0.01)
Q1, Q3 = quantiles[0], quantiles[1]

# calculate IQR, Upper Bound, and Lower Bound
IQR = Q3 - Q1
upper_bound = Q3 + (1.5 * IQR)
lower_bound = Q1 - (1.5 * IQR)
logger.info(f"statistical upper Bound: ${upper_bound:.2f}")
logger.info(f"statistical lower Bound: ${lower_bound:.2f}")
# apply the two-sided business logic tags
df_cart = df_cart.withColumn(
"order_value_category",
when(col("total") > upper_bound, "HIGH_VALUE")
.when(col("total") < lower_bound, "LOW_VALUE")
.otherwise("STANDARD_VALUE")
)

INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:SilverLayer:statistical upper Bound: $1201.90
INFO:SilverLayer:statistical lower Bound: $-669.97
INFO:py4j.clientserver:Received command c on object id p0


### 5. Pipeline Data Profiler: Discount Hypothesis Testing
We prepare an A/B test summary  for promotional strategy. 


In [0]:
from pyspark.sql.functions import avg, count, stddev, round
logger.info("Generating A/B Test metrics for Discount impact...")
# group by Discount vs. No Discount
df_hyp = df_cart.withColumn("has_discount", when(col("discount_percentage") > 0, "Yes").otherwise("No"))
# calculate Sample Size, Mean, and Standard Deviation 
df_test_results = df_hyp.groupBy("has_discount").agg(
count("*").alias("sample_size"),
round(avg("quantity"), 2).alias("mean_quantity"),
round(stddev("quantity"), 2).alias("std_deviation")
)
display(df_test_results)

INFO:SilverLayer:Generating A/B Test metrics for Discount impact...
INFO:py4j.clientserver:Received command c on object id p0


has_discount,sample_size,mean_quantity,std_deviation
No,1,2.0,null
Yes,121,3.31,1.34


###  Load to Silver (History-Preserving UPSERT)
We use Delta Lake's `MERGE` functionality. 
**Important:** We deliberately exclude `ingest_time` from the `UPDATE SET` clause. This ensures that if a product's price changes, we update the price, but permanently preserve the original timestamp of when the product first entered our ecosystem.

In [0]:
# Check if tables exist in the catalog
cart_exists = spark.catalog.tableExists("ecom_catalog.silver.cart_silver")
prod_exists = spark.catalog.tableExists("ecom_catalog.silver.products_silver")

try:
    if not cart_exists:
        logger.info("initializing cart_silver table")
        df_cart.write.format("delta").mode("overwrite").saveAsTable("ecom_catalog.silver.cart_silver")
    else:
        logger.info("merging into cart_silver..")
        df_cart.createOrReplaceTempView("cart_updates")# create a temp view to use in upsert
        
        # merge on compound key.dont update ingest_time.
        spark.sql("""
        MERGE INTO ecom_catalog.silver.cart_silver t
        USING cart_updates s
        ON t.cart_id = s.cart_id AND t.product_id = s.product_id
        WHEN MATCHED THEN UPDATE SET 
        t.user_id = s.user_id,
        t.product_title = s.product_title,
        t.quantity = s.quantity,
        t.price = s.price,
        t.total = s.total,
        t.discount_percentage = s.discount_percentage,
        t.discounted_total = s.discounted_total,
        t.order_value_category = s.order_value_category
        WHEN NOT MATCHED THEN INSERT *
        """)
  # product table upsert
    if not prod_exists:
        logger.info("initializing products_silver table.")
        df_products.write.format("delta").mode("overwrite").saveAsTable("ecom_catalog.silver.products_silver")
    else:
        logger.info("Performing SCD Type 2 Merge into products_silver.")
        
        # Create a temporary view of our incoming data
        df_products.createOrReplaceTempView("prod_updates") 
        # the SCD Type 2 Union Trick
        spark.sql("""
        MERGE INTO ecom_catalog.silver.products_silver target
        USING (
            --standard incoming data
            SELECT product_id as merge_key, * FROM prod_updates
            UNION ALL
             -- insert for a price change
            SELECT NULL as merge_key, updates.* FROM prod_updates updates 
            JOIN ecom_catalog.silver.products_silver target ON updates.product_id = target.product_id 
            WHERE target.is_current = True AND updates.price <> target.price
        ) staged_updates
        ON target.product_id = staged_updates.merge_key
        -- if price changed, expire the old record
        WHEN MATCHED AND target.is_current = True AND target.price <> staged_updates.price THEN 
          UPDATE SET target.is_current = False, target.valid_to = current_timestamp()
        -- insert the brand new record (Initial load or new price row)
        WHEN NOT MATCHED THEN   --insert  all as it is when not matched
          INSERT *
        """)
        logger.info("SCD Type 2 Merge completed for products_silver.")
except Exception as e:
    logger.error(f"Silver layer merge failed: {e}")
    raise

INFO:py4j.clientserver:Received command c on object id p1
INFO:py4j.clientserver:Received command c on object id p0
INFO:SilverLayer:merging into cart_silver..
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:SilverLayer:Performing SCD Type 2 Merge into products_silver.
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:py4j.clientserver:Received command c on object id p0
INFO:SilverLayer:SCD Type 2 Merge completed for products_silver.


In [0]:
%sql
SELECT * FROM cart_silver

cart_id,user_id,product_id,product_title,quantity,price,total,discount_percentage,discounted_total,ingest_time,order_value_category
24,6,134,Vivo S1,4,249.99,999.96,5.64,943.56,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
29,170,80,Huawei Matebook X Pro,2,1399.99,2799.98,9.99,2520.26,2026-04-05T08:06:19.077604Z,HIGH_VALUE
19,59,34,Nescafe Coffee,3,7.99,23.97,8.31,21.98,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
11,172,74,Spoon,3,4.99,14.97,2.78,14.55,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
30,177,171,Pacifica Touring,4,31999.99,127999.96,7.4,118527.96,2026-04-05T08:06:19.077604Z,HIGH_VALUE
9,194,178,Corset Leather With Skirt,2,89.99,179.98,12.59,157.32,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
20,90,90,Puma Future Rider Trainers,5,89.99,449.95,14.7,383.81,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
27,75,83,Blue & Black Check Shirt,5,29.99,149.95,8.76,136.81,2026-04-05T08:06:19.077604Z,STANDARD_VALUE
6,150,97,Rolex Datejust,3,10999.99,32999.97,10.58,29508.57,2026-04-05T08:06:19.077604Z,HIGH_VALUE
1001,404,9999,Data Engineer Coffee Mug,2,15.99,31.98,0.0,31.98,2026-04-05T08:08:59.608395Z,STANDARD_VALUE


In [0]:
%sql
select * from products_silver

product_id,title,description,brand,category,price,stock,rating,valid_from,valid_to,is_current
21,Cucumber,"Crisp and hydrating cucumbers, ideal for salads, snacks, or as a refreshing side.",Unknown,groceries,1.49,84,4.07,2026-04-05T08:06:21.234071Z,null,true
24,Fish Steak,"Quality fish steak, suitable for grilling, baking, or pan-searing.",Unknown,groceries,14.99,74,3.78,2026-04-05T08:06:21.234071Z,null,true
10,Gucci Bloom Eau de,"Gucci Bloom by Gucci is a floral and captivating fragrance, with notes of tuberose, jasmine, and Rangoon creeper. It's a modern and romantic scent.",Gucci,fragrances,79.99,91,2.74,2026-04-05T08:06:21.234071Z,null,true
11,Annibale Colombo Bed,"The Annibale Colombo Bed is a luxurious and elegant bed frame, crafted with high-quality materials for a comfortable and stylish bedroom.",Annibale Colombo,furniture,1899.99,88,4.77,2026-04-05T08:06:21.234071Z,null,true
3,Powder Canister,"The Powder Canister is a finely milled setting powder designed to set makeup and control shine. With a lightweight and translucent formula, it provides a smooth and matte finish.",Velvet Touch,beauty,14.99,89,4.64,2026-04-05T08:06:21.234071Z,null,true
17,Beef Steak,"High-quality beef steak, great for grilling or cooking to your preferred level of doneness.",Unknown,groceries,12.99,86,4.47,2026-04-05T08:06:21.234071Z,null,true
28,Ice Cream,"Creamy and delicious ice cream, available in various flavors for a delightful treat.",Unknown,groceries,5.49,27,3.39,2026-04-05T08:06:21.234071Z,null,true
18,Cat Food,Nutritious cat food formulated to meet the dietary needs of your feline friend.,Unknown,groceries,8.99,46,3.13,2026-04-05T08:06:21.234071Z,null,true
6,Calvin Klein CK One,"CK One by Calvin Klein is a classic unisex fragrance, known for its fresh and clean scent. It's a versatile fragrance suitable for everyday wear.",Calvin Klein,fragrances,49.99,29,4.37,2026-04-05T08:06:21.234071Z,null,true
12,Annibale Colombo Sofa,"The Annibale Colombo Sofa is a sophisticated and comfortable seating option, featuring exquisite design and premium upholstery for your living room.",Annibale Colombo,furniture,2499.99,60,3.92,2026-04-05T08:06:21.234071Z,null,true
